# Capstone project 2

Name: Bhushan Mandekar

Batch: 14

#### Time series forecasting: Daily minimum temperature

Research question: Can historical daily minimum temperature observations be used to accurately forecast future daily minimum temperatures in Melbourne?

 Import libraries, Load dataset and EDA

In [ ]:
# ============================================
# CELL 1: IMPORT LIBRARIES
# ============================================

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats

from statsmodels.tsa.stattools import (
    adfuller,
    acf,
    pacf
)

from statsmodels.tsa.seasonal import seasonal_decompose

from statsmodels.graphics.tsaplots import (
    plot_acf,
    plot_pacf
)

import warnings

warnings.filterwarnings("ignore")

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

# Plot settings
sns.set_theme(style="whitegrid")

print("Libraries imported successfully.")

In [ ]:
# Load the dataset and ignore extra blank lines
df = pd.read_csv(
    "daily-minimum-temperatures-in-me.csv",
    skip_blank_lines=True,
    on_bad_lines="skip"
)

# Remove rows where the date column contains no valid date
df = df[pd.to_datetime(df["Date"], errors="coerce").notna()].copy()

# Convert columns to the correct data types
df["Date"] = pd.to_datetime(df["Date"])
df["Daily minimum temperatures in Melbourne, Australia, 1981-1990"] = pd.to_numeric(
    df["Daily minimum temperatures in Melbourne, Australia, 1981-1990"],
    errors="coerce"
)

# Remove any rows with missing dates or temperatures
df = df.dropna().reset_index(drop=True)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)
print("\nFirst 5 rows:")
display(df.head())

print("\nLast 5 rows:")
display(df.tail())

In [ ]:
# ============================================
# INITIAL INSPECTION
# ============================================

print("Column names:")
print(df.columns.tolist())

print("\nFirst 10 rows:")
display(df.head(10))

print("\nLast 10 rows:")
display(df.tail(10))

In [ ]:
# ============================================
# DATA TYPES
# ============================================

print(df.dtypes)

In [ ]:
# ============================================
# RENAME COLUMNS
# ============================================

df.columns = ["Date", "Temperature"]

print(df.columns.tolist())

In [ ]:
# ============================================
# CHECK TEMPERATURE VALUES
# ============================================

print(df["Temperature"].head(20).to_list())

In [ ]:
# Try converting to numeric temporarily
temp_test = pd.to_numeric(
    df["Temperature"],
    errors="coerce"
)

invalid_values = df.loc[
    temp_test.isna(),
    "Temperature"
]

print("Invalid temperature values:")
print(invalid_values)

In [ ]:
# ============================================
# CLEAN TEMPERATURE
# ============================================

df["Temperature"] = (
    df["Temperature"]
    .astype(str)
    .str.replace("?", "", regex=False)
)

df["Temperature"] = pd.to_numeric(
    df["Temperature"],
    errors="coerce"
)

print(df.dtypes)

In [ ]:
# ============================================
# CONVERT DATE
# ============================================

df["Date"] = pd.to_datetime(
    df["Date"],
    errors="coerce"
)

print(df.dtypes)

In [ ]:
print(df["Date"].head())

In [ ]:
# ============================================
# MISSING VALUES
# ============================================

missing_summary = pd.DataFrame({
    "Missing_Count": df.isnull().sum(),
    "Missing_Percentage": (
        df.isnull().mean() * 100
    )
})

display(missing_summary)

In [ ]:
# ============================================
# MISSING OBSERVATIONS
# ============================================

missing_rows = df[
    df["Date"].isna() |
    df["Temperature"].isna()
]

display(missing_rows)

In [ ]:
# ============================================
# DUPLICATE ROWS
# ============================================

duplicate_rows = df.duplicated().sum()

print(
    "Number of duplicate rows:",
    duplicate_rows
)

In [ ]:
# ============================================
# DUPLICATE DATES
# ============================================

duplicate_dates = df["Date"].duplicated().sum()

print(
    "Number of duplicate dates:",
    duplicate_dates
)

In [ ]:
df[
    df["Date"].duplicated(keep=False)
]

In [ ]:
# ============================================
# SORT BY DATE
# ============================================

df = df.sort_values(
    "Date"
).reset_index(drop=True)

print(df.head())

In [ ]:
# ============================================
# DATE RANGE
# ============================================

print(
    "Start date:",
    df["Date"].min()
)

print(
    "End date:",
    df["Date"].max()
)

print(
    "Number of observations:",
    len(df)
)

In [ ]:
# ============================================
# CHECK MISSING DATES
# ============================================

expected_dates = pd.date_range(
    start=df["Date"].min(),
    end=df["Date"].max(),
    freq="D"
)

missing_dates = expected_dates.difference(
    df["Date"]
)

print(
    "Number of missing dates:",
    len(missing_dates)
)

if len(missing_dates) > 0:
    print("\nMissing dates:")
    print(missing_dates)

In [ ]:
# ============================================
# SET DATE AS INDEX
# ============================================

df = df.set_index("Date")

print(df.head())

In [ ]:
# ============================================
# SET DAILY FREQUENCY
# ============================================

df = df.asfreq("D")

print(
    "Inferred frequency:",
    pd.infer_freq(df.index)
)

In [ ]:
# ============================================
# CHECK MISSING VALUES AGAIN
# ============================================

print(df.isnull().sum())

In [ ]:
# ============================================
# INTERPOLATE MISSING TEMPERATURES
# ============================================

df["Temperature"] = (
    df["Temperature"]
    .interpolate(method="linear")
)

print("Missing values after interpolation:")
print(df.isnull().sum())

In [ ]:
# ============================================
# DATA QUALITY CHECK
# ============================================

print("Shape:", df.shape)

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDate range:")
print(df.index.min(), "to", df.index.max())

print("\nFrequency:")
print(pd.infer_freq(df.index))

In [ ]:
# ============================================
# DESCRIPTIVE STATISTICS
# ============================================

display(
    df["Temperature"].describe()
)

In [ ]:
temperature_stats = pd.Series({
    "Mean": df["Temperature"].mean(),
    "Median": df["Temperature"].median(),
    "Standard Deviation": df["Temperature"].std(),
    "Variance": df["Temperature"].var(),
    "Minimum": df["Temperature"].min(),
    "Maximum": df["Temperature"].max(),
    "Range": (
        df["Temperature"].max()
        - df["Temperature"].min()
    ),
    "Skewness": df["Temperature"].skew(),
    "Kurtosis": df["Temperature"].kurtosis()
})

display(
    temperature_stats.to_frame(
        name="Value"
    )
)

In [ ]:
# ============================================
# TIME SERIES PLOT
# ============================================

plt.figure(figsize=(16, 6))

plt.plot(
    df.index,
    df["Temperature"],
    linewidth=0.8
)

plt.title(
    "Daily Minimum Temperature Over Time",
    fontsize=16
)

plt.xlabel("Date")
plt.ylabel("Minimum Temperature (°C)")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# TEMPERATURE DISTRIBUTION
# ============================================

plt.figure(figsize=(10, 6))

sns.histplot(
    df["Temperature"],
    bins=30,
    kde=True
)

plt.title(
    "Distribution of Daily Minimum Temperature",
    fontsize=15
)

plt.xlabel("Minimum Temperature (°C)")
plt.ylabel("Frequency")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# BOXPLOT
# ============================================

plt.figure(figsize=(8, 5))

sns.boxplot(
    y=df["Temperature"]
)

plt.title(
    "Boxplot of Daily Minimum Temperature"
)

plt.ylabel("Minimum Temperature (°C)")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# OUTLIER DETECTION
# ============================================

Q1 = df["Temperature"].quantile(0.25)
Q3 = df["Temperature"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[
    (df["Temperature"] < lower_bound) |
    (df["Temperature"] > upper_bound)
]

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)

print("\nLower bound:", lower_bound)
print("Upper bound:", upper_bound)

print(
    "\nNumber of potential outliers:",
    len(outliers)
)

In [ ]:
display(outliers.head(20))

In [ ]:
# ============================================
# CALENDAR VARIABLES
# ============================================

df["Year"] = df.index.year
df["Month"] = df.index.month
df["Month_Name"] = df.index.month_name()
df["Day_of_Year"] = df.index.dayofyear

print(df.head())

In [ ]:
# ============================================
# MONTHLY AVERAGE TEMPERATURE
# ============================================

monthly_avg = (
    df.groupby("Month")["Temperature"]
    .mean()
)

display(monthly_avg)

In [ ]:
# ============================================
# MONTHLY SEASONAL PATTERN
# ============================================

plt.figure(figsize=(10, 5))

plt.plot(
    monthly_avg.index,
    monthly_avg.values,
    marker="o"
)

plt.xticks(range(1, 13))

plt.title(
    "Average Minimum Temperature by Month",
    fontsize=15
)

plt.xlabel("Month")
plt.ylabel("Average Minimum Temperature (°C)")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# MONTHLY BOXPLOTS
# ============================================

plt.figure(figsize=(14, 6))

sns.boxplot(
    x=df["Month"],
    y=df["Temperature"]
)

plt.title(
    "Distribution of Daily Minimum Temperature by Month"
)

plt.xlabel("Month")
plt.ylabel("Minimum Temperature (°C)")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# YEARLY AVERAGE TEMPERATURE
# ============================================

yearly_avg = (
    df.groupby("Year")["Temperature"]
    .mean()
)

display(yearly_avg)

In [ ]:
# ============================================
# YEARLY AVERAGE TEMPERATURE PLOT
# ============================================

plt.figure(figsize=(12, 5))

plt.plot(
    yearly_avg.index,
    yearly_avg.values,
    marker="o"
)

plt.title(
    "Annual Average Minimum Temperature"
)

plt.xlabel("Year")
plt.ylabel("Average Minimum Temperature (°C)")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# YEAR-MONTH HEATMAP
# ============================================

monthly_year = df.pivot_table(
    values="Temperature",
    index="Year",
    columns="Month",
    aggfunc="mean"
)

plt.figure(figsize=(14, 7))

sns.heatmap(
    monthly_year,
    annot=True,
    fmt=".1f",
    cmap="coolwarm"
)

plt.title(
    "Monthly Average Minimum Temperature by Year"
)

plt.xlabel("Month")
plt.ylabel("Year")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# SEASONAL CLASSIFICATION
# ============================================

def get_season(month):

    if month in [12, 1, 2]:
        return "Summer"

    elif month in [3, 4, 5]:
        return "Autumn"

    elif month in [6, 7, 8]:
        return "Winter"

    else:
        return "Spring"


df["Season"] = df["Month"].apply(
    get_season
)

print(
    df["Season"].value_counts()
)

In [ ]:
# ============================================
# SEASONAL STATISTICS
# ============================================

season_summary = (
    df.groupby("Season")["Temperature"]
    .agg([
        "count",
        "mean",
        "median",
        "std",
        "min",
        "max"
    ])
)

display(season_summary)

In [ ]:
# ============================================
# SEASONAL BOXPLOT
# ============================================

plt.figure(figsize=(10, 6))

sns.boxplot(
    x="Season",
    y="Temperature",
    data=df
)

plt.title(
    "Distribution of Daily Minimum Temperature by Season"
)

plt.xlabel("Season")
plt.ylabel("Minimum Temperature (°C)")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# ROLLING AVERAGES
# ============================================

df["Rolling_7"] = (
    df["Temperature"]
    .rolling(window=7)
    .mean()
)

df["Rolling_30"] = (
    df["Temperature"]
    .rolling(window=30)
    .mean()
)

In [ ]:
# ============================================
# ROLLING AVERAGES PLOT
# ============================================

plt.figure(figsize=(16, 6))

plt.plot(
    df.index,
    df["Temperature"],
    alpha=0.25,
    linewidth=0.6,
    label="Daily Temperature"
)

plt.plot(
    df.index,
    df["Rolling_7"],
    linewidth=1.2,
    label="7-Day Rolling Mean"
)

plt.plot(
    df.index,
    df["Rolling_30"],
    linewidth=1.5,
    label="30-Day Rolling Mean"
)

plt.title(
    "Daily Temperature with Rolling Averages"
)

plt.xlabel("Date")
plt.ylabel("Temperature (°C)")

plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# ROLLING STANDARD DEVIATION
# ============================================

df["Rolling_STD_30"] = (
    df["Temperature"]
    .rolling(window=30)
    .std()
)

plt.figure(figsize=(16, 5))

plt.plot(
    df.index,
    df["Rolling_STD_30"]
)

plt.title(
    "30-Day Rolling Standard Deviation"
)

plt.xlabel("Date")
plt.ylabel("Standard Deviation (°C)")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# SEASONAL DECOMPOSITION
# ============================================

decomposition = seasonal_decompose(
    df["Temperature"],
    model="additive",
    period=365
)

fig = decomposition.plot()

fig.set_size_inches(16, 12)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# DAY-OF-YEAR SEASONAL PATTERN
# ============================================

daily_seasonal_pattern = (
    df.groupby("Day_of_Year")["Temperature"]
    .mean()
)

plt.figure(figsize=(16, 5))

plt.plot(
    daily_seasonal_pattern.index,
    daily_seasonal_pattern.values
)

plt.title(
    "Average Minimum Temperature Across the Year"
)

plt.xlabel("Day of Year")
plt.ylabel("Average Minimum Temperature (°C)")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# AUTOCORRELATION
# ============================================

fig, ax = plt.subplots(
    figsize=(14, 5)
)

plot_acf(
    df["Temperature"],
    lags=60,
    ax=ax
)

ax.set_title(
    "Autocorrelation of Daily Minimum Temperature"
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# PARTIAL AUTOCORRELATION
# ============================================

fig, ax = plt.subplots(
    figsize=(14, 5)
)

plot_pacf(
    df["Temperature"],
    lags=60,
    ax=ax,
    method="ywm"
)

ax.set_title(
    "Partial Autocorrelation of Daily Minimum Temperature"
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# IMPORTANT LAG CORRELATIONS
# ============================================

lags = [
    1,
    2,
    3,
    7,
    14,
    30,
    90,
    180,
    365
]

lag_results = []

for lag in lags:

    correlation = (
        df["Temperature"]
        .autocorr(lag=lag)
    )

    lag_results.append({
        "Lag": lag,
        "Autocorrelation": correlation
    })

lag_results = pd.DataFrame(
    lag_results
)

display(lag_results)

In [ ]:
# ============================================
# ADF TEST
# ============================================

temperature_series = (
    df["Temperature"]
    .dropna()
)

adf_result = adfuller(
    temperature_series
)

print(
    "ADF Statistic:",
    adf_result[0]
)

print(
    "p-value:",
    adf_result[1]
)

print("\nCritical Values:")

for key, value in adf_result[4].items():

    print(
        f"{key}: {value}"
    )

In [ ]:
if adf_result[1] < 0.05:
    print("The series is likely stationary.")
else:
    print("The series is likely non-stationary.")

In [ ]:
# ============================================
# FIRST DIFFERENCE
# ============================================

df["Temperature_Diff"] = (
    df["Temperature"]
    .diff()
)

plt.figure(figsize=(16, 5))

plt.plot(
    df.index,
    df["Temperature_Diff"]
)

plt.title(
    "First Difference of Daily Minimum Temperature"
)

plt.xlabel("Date")
plt.ylabel("Differenced Temperature")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# ADF TEST AFTER DIFFERENCING
# ============================================

diff_series = (
    df["Temperature_Diff"]
    .dropna()
)

adf_diff_result = adfuller(
    diff_series
)

print(
    "ADF Statistic:",
    adf_diff_result[0]
)

print(
    "p-value:",
    adf_diff_result[1]
)

print("\nCritical Values:")

for key, value in adf_diff_result[4].items():

    print(
        f"{key}: {value}"
    )

In [ ]:
# ============================================
# ACF OF DIFFERENCED SERIES
# ============================================

fig, ax = plt.subplots(
    figsize=(14, 5)
)

plot_acf(
    diff_series,
    lags=60,
    ax=ax
)

ax.set_title(
    "ACF of First-Differenced Temperature"
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# PACF OF DIFFERENCED SERIES
# ============================================

fig, ax = plt.subplots(
    figsize=(14, 5)
)

plot_pacf(
    diff_series,
    lags=60,
    ax=ax,
    method="ywm"
)

ax.set_title(
    "PACF of First-Differenced Temperature"
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# ANNUAL LAG
# ============================================

df["Lag_365"] = (
    df["Temperature"]
    .shift(365)
)

annual_corr = (
    df["Temperature"]
    .corr(df["Lag_365"])
)

print(
    "Correlation with temperature 365 days earlier:",
    annual_corr
)

In [ ]:
# ============================================
# WEEKLY LAG
# ============================================

weekly_corr = (
    df["Temperature"]
    .autocorr(lag=7)
)

print(
    "Lag-7 autocorrelation:",
    weekly_corr
)

In [ ]:
# ============================================
# COLDEST DAYS
# ============================================

coldest_days = (
    df[["Temperature"]]
    .sort_values("Temperature")
    .head(10)
)

display(coldest_days)

In [ ]:
# ============================================
# WARMEST DAYS
# ============================================

warmest_days = (
    df[["Temperature"]]
    .sort_values(
        "Temperature",
        ascending=False
    )
    .head(10)
)

display(warmest_days)

In [ ]:
# ============================================
# YEARLY EXTREMES
# ============================================

yearly_extremes = (
    df.groupby("Year")["Temperature"]
    .agg(
        Mean="mean",
        Minimum="min",
        Maximum="max",
        SD="std"
    )
)

display(yearly_extremes)

In [ ]:
# ============================================
# FINAL MODELING DATASET
# ============================================

model_df = df[
    ["Temperature"]
].copy()

print(model_df.head())

print("\nShape:")
print(model_df.shape)

print("\nMissing values:")
print(model_df.isnull().sum())

In [ ]:
# ============================================
# SAVE CLEAN DATA
# ============================================

model_df.to_csv(
    "daily_minimum_temperature_cleaned.csv"
)

print(
    "Cleaned dataset saved successfully."
)

### Final EDA summary

In [ ]:
print("=" * 60)
print("FINAL DATASET SUMMARY")
print("=" * 60)

print(
    f"Number of observations: {len(model_df)}"
)

print(
    f"Start date: {model_df.index.min().date()}"
)

print(
    f"End date: {model_df.index.max().date()}"
)

print(
    f"Frequency: {pd.infer_freq(model_df.index)}"
)

print(
    f"Mean temperature: "
    f"{model_df['Temperature'].mean():.2f} °C"
)

print(
    f"Minimum temperature: "
    f"{model_df['Temperature'].min():.2f} °C"
)

print(
    f"Maximum temperature: "
    f"{model_df['Temperature'].max():.2f} °C"
)

print(
    f"Standard deviation: "
    f"{model_df['Temperature'].std():.2f} °C"
)

print(
    f"Missing values: "
    f"{model_df['Temperature'].isna().sum()}"
)

print("=" * 60)

## Forecasting

In [ ]:
# ============================================
# IMPORT LIBRARIES
# ============================================

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX

import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid")

In [ ]:
# ============================================
# LOAD CLEANED DATA
# ============================================

file_path = "daily_minimum_temperature_cleaned.csv"

df = pd.read_csv(
    file_path,
    parse_dates=["Date"],
    index_col="Date"
)

df.head()

In [ ]:
print(df.shape)
print(df.index.min())
print(df.index.max())
print(df.isnull().sum())

In [ ]:
# ============================================
# TARGET SERIES
# ============================================

temperature = df["Temperature"].copy()

print(temperature.head())

In [ ]:
print("Frequency:", pd.infer_freq(temperature.index))

In [ ]:
# ============================================
# TRAIN / TEST SPLIT
# ============================================

test_size = 365

train = temperature.iloc[:-test_size]
test = temperature.iloc[-test_size:]

print("Training observations:", len(train))
print("Testing observations:", len(test))

print("\nTraining period:")
print(train.index.min(), "to", train.index.max())

print("\nTesting period:")
print(test.index.min(), "to", test.index.max())

In [ ]:
# ============================================
# TRAIN / TEST VISUALIZATION
# ============================================

plt.figure(figsize=(16, 6))

plt.plot(
    train.index,
    train,
    label="Training Data"
)

plt.plot(
    test.index,
    test,
    label="Test Data"
)

plt.axvline(
    test.index[0],
    linestyle="--",
    label="Train/Test Split"
)

plt.title(
    "Training and Testing Data"
)

plt.xlabel("Date")
plt.ylabel("Minimum Temperature (°C)")

plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# EVALUATION FUNCTION
# ============================================

def evaluate_model(actual, predicted):

    mae = mean_absolute_error(
        actual,
        predicted
    )

    rmse = np.sqrt(
        mean_squared_error(
            actual,
            predicted
        )
    )

    return {
        "MAE": mae,
        "RMSE": rmse
    }

In [ ]:
# ============================================
# NAÏVE FORECAST
# ============================================

naive_forecast = np.repeat(
    train.iloc[-1],
    len(test)
)

naive_forecast = pd.Series(
    naive_forecast,
    index=test.index
)

naive_metrics = evaluate_model(
    test,
    naive_forecast
)

print("Naïve Model Results")
print("-------------------")
print("MAE :", round(naive_metrics["MAE"], 4))
print("RMSE:", round(naive_metrics["RMSE"], 4))

In [ ]:
# ============================================
# NAÏVE FORECAST PLOT
# ============================================

plt.figure(figsize=(16, 6))

plt.plot(
    test.index,
    test,
    label="Actual"
)

plt.plot(
    test.index,
    naive_forecast,
    label="Naïve Forecast"
)

plt.title(
    "Naïve Forecast vs Actual Temperature"
)

plt.xlabel("Date")
plt.ylabel("Temperature (°C)")

plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# SEASONAL NAÏVE FORECAST
# ============================================

seasonal_naive_forecast = train.iloc[-365:].values

seasonal_naive_forecast = pd.Series(
    seasonal_naive_forecast,
    index=test.index
)

seasonal_naive_metrics = evaluate_model(
    test,
    seasonal_naive_forecast
)

print("Seasonal Naïve Model Results")
print("----------------------------")
print(
    "MAE :",
    round(seasonal_naive_metrics["MAE"], 4)
)

print(
    "RMSE:",
    round(seasonal_naive_metrics["RMSE"], 4)
)

In [ ]:
# ============================================
# SEASONAL NAÏVE PLOT
# ============================================

plt.figure(figsize=(16, 6))

plt.plot(
    test.index,
    test,
    label="Actual"
)

plt.plot(
    test.index,
    seasonal_naive_forecast,
    label="Seasonal Naïve Forecast"
)

plt.title(
    "Seasonal Naïve Forecast vs Actual"
)

plt.xlabel("Date")
plt.ylabel("Temperature (°C)")

plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# BASELINE MODEL COMPARISON
# ============================================

baseline_results = pd.DataFrame({
    "Model": [
        "Naïve",
        "Seasonal Naïve"
    ],

    "MAE": [
        naive_metrics["MAE"],
        seasonal_naive_metrics["MAE"]
    ],

    "RMSE": [
        naive_metrics["RMSE"],
        seasonal_naive_metrics["RMSE"]
    ]
})

baseline_results

In [ ]:
# ============================================
# ARIMA MODEL
# ============================================

arima_model = ARIMA(
    train,
    order=(1, 0, 1)
)

arima_fit = arima_model.fit()

print(arima_fit.summary())

In [ ]:
# ============================================
# ARIMA FORECAST
# ============================================

arima_forecast = arima_fit.forecast(
    steps=len(test)
)

arima_forecast.index = test.index

arima_metrics = evaluate_model(
    test,
    arima_forecast
)

print("ARIMA(1,0,1)")
print("------------")
print(
    "MAE :",
    round(arima_metrics["MAE"], 4)
)

print(
    "RMSE:",
    round(arima_metrics["RMSE"], 4)
)

In [ ]:
# ============================================
# ARIMA FORECAST PLOT
# ============================================

plt.figure(figsize=(16, 6))

plt.plot(
    test.index,
    test,
    label="Actual"
)

plt.plot(
    test.index,
    arima_forecast,
    label="ARIMA Forecast"
)

plt.title(
    "ARIMA Forecast vs Actual Temperature"
)

plt.xlabel("Date")
plt.ylabel("Temperature (°C)")

plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# ARIMA MODEL SEARCH
# ============================================

import itertools

p_values = range(0, 4)
d_values = range(0, 2)
q_values = range(0, 4)

arima_results = []

for p, d, q in itertools.product(
    p_values,
    d_values,
    q_values
):

    try:

        model = ARIMA(
            train,
            order=(p, d, q)
        )

        fitted = model.fit()

        forecast = fitted.forecast(
            steps=len(test)
        )

        forecast.index = test.index

        metrics = evaluate_model(
            test,
            forecast
        )

        arima_results.append({
            "Model": f"ARIMA({p},{d},{q})",
            "p": p,
            "d": d,
            "q": q,
            "AIC": fitted.aic,
            "BIC": fitted.bic,
            "MAE": metrics["MAE"],
            "RMSE": metrics["RMSE"]
        })

    except Exception as e:
        continue

arima_results = pd.DataFrame(
    arima_results
)

arima_results = arima_results.sort_values(
    "RMSE"
)

display(
    arima_results.head(10)
)

Best model based on test-set RMSE is:
ARIMA(1,0,2)

All the top models have: d=0
So the ARIMA search is suggesting that differencing is not necessary for these models.

In [ ]:
# ============================================
# HOLT-WINTERS / EXPONENTIAL SMOOTHING
# ============================================

hw_model = ExponentialSmoothing(
    train,
    trend="add",
    seasonal="add",
    seasonal_periods=365
)

hw_fit = hw_model.fit(
    optimized=True
)

print(hw_fit.summary())

In [ ]:
# ============================================
# HOLT-WINTERS FORECAST
# ============================================

hw_forecast = hw_fit.forecast(
    steps=len(test)
)

hw_forecast.index = test.index

hw_metrics = evaluate_model(
    test,
    hw_forecast
)

print("Holt-Winters Results")
print("--------------------")

print(
    "MAE :",
    round(hw_metrics["MAE"], 4)
)

print(
    "RMSE:",
    round(hw_metrics["RMSE"], 4)
)

In [ ]:
# ============================================
# HOLT-WINTERS FORECAST PLOT
# ============================================

plt.figure(figsize=(16, 6))

plt.plot(
    test.index,
    test,
    label="Actual"
)

plt.plot(
    test.index,
    hw_forecast,
    label="Holt-Winters Forecast"
)

plt.title(
    "Holt-Winters Forecast vs Actual Temperature"
)

plt.xlabel("Date")
plt.ylabel("Minimum Temperature (°C)")

plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# ARIMA VS HOLT-WINTERS
# ============================================

comparison = pd.DataFrame({
    "Model": [
        "Naïve",
        "Seasonal Naïve",
        "ARIMA(1,0,2)",
        "Holt-Winters"
    ],

    "MAE": [
        naive_metrics["MAE"],
        seasonal_naive_metrics["MAE"],
        arima_metrics["MAE"],
        hw_metrics["MAE"]
    ],

    "RMSE": [
        naive_metrics["RMSE"],
        seasonal_naive_metrics["RMSE"],
        arima_metrics["RMSE"],
        hw_metrics["RMSE"]
    ]
})

comparison = comparison.sort_values(
    "RMSE"
)

display(comparison)

conventional SARIMA would require something like:
SARIMA(p,d,q)(P,D,Q)_365
A 365-day seasonal ARIMA can be computationally expensive and can require a very large number of parameters.
Instead, we can represent the annual seasonal cycle using Fourier terms.
This allows the model to learn the annual temperature cycle.

In [ ]:
# ============================================
# FOURIER TERMS
# ============================================

def create_fourier_terms(index, period=365.25, K=5):

    t = np.arange(len(index))

    fourier = pd.DataFrame(
        index=index
    )

    for k in range(1, K + 1):

        fourier[f"sin_{k}"] = np.sin(
            2 * np.pi * k * t / period
        )

        fourier[f"cos_{k}"] = np.cos(
            2 * np.pi * k * t / period
        )

    return fourier

In [ ]:
# ============================================
# GENERATE FOURIER FEATURES
# ============================================

fourier = create_fourier_terms(
    temperature.index,
    period=365.25,
    K=5
)

print(
    fourier.head()
)

print(
    "\nShape:",
    fourier.shape
)

In [ ]:
# ============================================
# TRAIN / TEST FOURIER FEATURES
# ============================================

X_train = fourier.loc[
    train.index
]

X_test = fourier.loc[
    test.index
]

print(
    "X_train shape:",
    X_train.shape
)

print(
    "X_test shape:",
    X_test.shape
)

In [ ]:
# ============================================
# SARIMAX + FOURIER
# ============================================

sarimax_model = SARIMAX(
    train,
    exog=X_train,
    order=(1, 0, 2),
    seasonal_order=(0, 0, 0, 0),
    enforce_stationarity=False,
    enforce_invertibility=False
)

sarimax_fit = sarimax_model.fit(
    disp=False
)

print(
    sarimax_fit.summary()
)

In [ ]:
# ============================================
# SARIMAX FORECAST
# ============================================

sarimax_forecast = sarimax_fit.forecast(
    steps=len(test),
    exog=X_test
)

sarimax_forecast.index = test.index

sarimax_metrics = evaluate_model(
    test,
    sarimax_forecast
)

print("SARIMAX + Fourier Results")
print("-------------------------")

print(
    "MAE :",
    round(sarimax_metrics["MAE"], 4)
)

print(
    "RMSE:",
    round(sarimax_metrics["RMSE"], 4)
)

In [ ]:
# ============================================
# SARIMAX FORECAST PLOT
# ============================================

plt.figure(figsize=(16, 6))

plt.plot(
    test.index,
    test,
    label="Actual"
)

plt.plot(
    test.index,
    sarimax_forecast,
    label="SARIMAX + Fourier"
)

plt.title(
    "SARIMAX + Fourier Forecast vs Actual"
)

plt.xlabel("Date")
plt.ylabel("Minimum Temperature (°C)")

plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# COMPLETE MODEL COMPARISON
# ============================================

comparison = pd.DataFrame({

    "Model": [
        "Naïve",
        "Seasonal Naïve",
        "ARIMA(1,0,2)",
        "Holt-Winters",
        "SARIMAX + Fourier"
    ],

    "MAE": [
        naive_metrics["MAE"],
        seasonal_naive_metrics["MAE"],
        arima_metrics["MAE"],
        hw_metrics["MAE"],
        sarimax_metrics["MAE"]
    ],

    "RMSE": [
        naive_metrics["RMSE"],
        seasonal_naive_metrics["RMSE"],
        arima_metrics["RMSE"],
        hw_metrics["RMSE"],
        sarimax_metrics["RMSE"]
    ]
})

comparison = comparison.sort_values(
    "RMSE"
).reset_index(drop=True)

display(comparison)

Since SARIMAX + Fourier is currently the best model, the next steps would focus on validating it properly and checking whether its residuals are acceptable.

Next step- Create a validation set
Instead of using the 1990 test year for model selection, we'll use:
- 1981–1988 → training
- 1989 → validation
- 1990 → final test

In [ ]:
print("Start:", temperature.index.min())
print("End:", temperature.index.max())

In [ ]:
train_cv = temperature[temperature.index < "1989-01-01"]

validation = temperature[
    (temperature.index >= "1989-01-01") &
    (temperature.index < "1990-01-01")
]

final_test = temperature[temperature.index >= "1990-01-01"]

print("Training:", train_cv.index.min(), "to", train_cv.index.max(), len(train_cv))
print("Validation:", validation.index.min(), "to", validation.index.max(), len(validation))
print("Final Test:", final_test.index.min(), "to", final_test.index.max(), len(final_test))

In [ ]:
def create_fourier_terms(index, period=365.25, K=5):
    t = np.arange(len(index))
    
    fourier = pd.DataFrame(index=index)
    
    for k in range(1, K + 1):
        fourier[f"sin_{k}"] = np.sin(2 * np.pi * k * t / period)
        fourier[f"cos_{k}"] = np.cos(2 * np.pi * k * t / period)
    
    return fourier

In [ ]:
fourier_all = create_fourier_terms(
    temperature.index,
    period=365.25,
    K=5
)

X_train_cv = fourier_all.loc[train_cv.index]
X_validation = fourier_all.loc[validation.index]
X_final_test = fourier_all.loc[final_test.index]

print(X_train_cv.shape)
print(X_validation.shape)
print(X_final_test.shape)

In [ ]:
# ============================================
# Fit SARIMAX MODEL FOR CROSS-VALIDATION
# ============================================

sarimax_cv = SARIMAX(
    train_cv,
    exog=X_train_cv,
    order=(1, 0, 2),
    seasonal_order=(0, 0, 0, 0),
    enforce_stationarity=False,
    enforce_invertibility=False
)

sarimax_cv_fit = sarimax_cv.fit(disp=False)

print(sarimax_cv_fit.summary())

In [ ]:
# ============================================ 
# Forecast the 1989 validation year 
# ============================================

validation_forecast = sarimax_cv_fit.forecast(
    steps=len(validation),
    exog=X_validation
)

validation_forecast.index = validation.index

In [ ]:
validation_metrics = evaluate_model(
    validation,
    validation_forecast
)

print("Validation performance:")
print(validation_metrics)

In [ ]:
plt.figure(figsize=(15, 5))

plt.plot(
    validation.index,
    validation,
    label="Actual"
)

plt.plot(
    validation_forecast.index,
    validation_forecast,
    label="SARIMAX + Fourier"
)

plt.title("SARIMAX + Fourier: Validation Forecast")
plt.xlabel("Date")
plt.ylabel("Minimum Temperature (°C)")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
validation_residuals = validation - validation_forecast

print(validation_residuals.describe())

In [ ]:
plt.figure(figsize=(15, 4))

plt.plot(
    validation_residuals.index,
    validation_residuals
)

plt.axhline(0, linestyle="--")

plt.title("Validation Forecast Residuals")
plt.xlabel("Date")
plt.ylabel("Residual")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# Residual diagnostics for the fitted model
# ============================================

residuals = sarimax_cv_fit.resid.dropna()

plt.figure(figsize=(15, 4))

plt.plot(residuals)

plt.axhline(0, linestyle="--")

plt.title("SARIMAX Residuals")
plt.xlabel("Date")
plt.ylabel("Residual")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.hist(
    residuals,
    bins=30,
    edgecolor="black"
)

plt.title("Distribution of SARIMAX Residuals")
plt.xlabel("Residual")
plt.ylabel("Frequency")

plt.tight_layout()
plt.show()

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

plt.figure(figsize=(12, 5))

plot_acf(
    residuals,
    lags=40,
    ax=plt.gca()
)

plt.title("ACF of SARIMAX Residuals")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 5))

plot_pacf(
    residuals,
    lags=40,
    method="ywm",
    ax=plt.gca()
)

plt.title("PACF of SARIMAX Residuals")

plt.tight_layout()
plt.show()

In [ ]:
from statsmodels.stats.diagnostic import acorr_ljungbox

ljung_box = acorr_ljungbox(
    residuals,
    lags=[7, 14, 30],
    return_df=True
)

ljung_box

In [ ]:
# Examine SARIMAX residuals

residuals = sarimax_cv_fit.resid.dropna()

print("Number of residuals:", len(residuals))
print("\nResidual statistics:")
print(residuals.describe())

In [ ]:
plt.figure(figsize=(15, 4))

plt.plot(residuals)

plt.axhline(0, linestyle="--")

plt.title("SARIMAX + Fourier Residuals")
plt.xlabel("Date")
plt.ylabel("Residual")

plt.tight_layout()
plt.show()

In [ ]:
# Check residual autocorrelation

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

fig, ax = plt.subplots(figsize=(12, 5))

plot_acf(
    residuals,
    lags=40,
    ax=ax
)

ax.set_title("ACF of SARIMAX + Fourier Residuals")

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

plot_pacf(
    residuals,
    lags=40,
    method="ywm",
    ax=ax
)

ax.set_title("PACF of SARIMAX + Fourier Residuals")

plt.tight_layout()
plt.show()

In [ ]:
from statsmodels.stats.diagnostic import acorr_ljungbox

ljung_box = acorr_ljungbox(
    residuals,
    lags=[7, 14, 30],
    return_df=True
)

print(ljung_box)

In [ ]:
import scipy.stats as stats

plt.figure(figsize=(8, 5))

stats.probplot(
    residuals,
    dist="norm",
    plot=plt
)

plt.title("Q-Q Plot of SARIMAX + Fourier Residuals")

plt.tight_layout()
plt.show()

SARIMAX + Fourier captures the annual seasonal pattern extremely well, but some short-term temporal dependence remains unexplained.

In [ ]:
sarimax_orders = [
    (1, 0, 1),
    (1, 0, 2),
    (1, 0, 3),
    (2, 0, 1),
    (2, 0, 2),
    (2, 0, 3),
    (3, 0, 1),
    (3, 0, 2),
    (3, 0, 3)
]

sarimax_results = []

for order in sarimax_orders:
    
    print(f"Fitting SARIMAX{order}...")
    
    try:
        model = SARIMAX(
            train_cv,
            exog=X_train_cv,
            order=order,
            seasonal_order=(0, 0, 0, 0),
            enforce_stationarity=False,
            enforce_invertibility=False
        )
        
        fit = model.fit(disp=False)
        
        forecast = fit.forecast(
            steps=len(validation),
            exog=X_validation
        )
        
        forecast.index = validation.index
        
        metrics = evaluate_model(
            validation,
            forecast
        )
        
        sarimax_results.append({
            "Order": str(order),
            "AIC": fit.aic,
            "BIC": fit.bic,
            "MAE": metrics["MAE"],
            "RMSE": metrics["RMSE"]
        })
        
    except Exception as e:
        print(f"Error for {order}: {e}")

sarimax_comparison = (
    pd.DataFrame(sarimax_results)
    .sort_values("RMSE")
    .reset_index(drop=True)
)

sarimax_comparison

In [ ]:
# Check residuals for the best candidate
best_order = (2, 0, 2)

best_sarimax = SARIMAX(
    train_cv,
    exog=X_train_cv,
    order=best_order,
    seasonal_order=(0, 0, 0, 0),
    enforce_stationarity=False,
    enforce_invertibility=False
)

best_sarimax_fit = best_sarimax.fit(disp=False)

best_residuals = best_sarimax_fit.resid.dropna()

print(best_sarimax_fit.summary())

In [ ]:
best_ljung_box = acorr_ljungbox(
    best_residuals,
    lags=[7, 14, 30],
    return_df=True
)

print(best_ljung_box)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

plot_acf(
    best_residuals,
    lags=40,
    ax=ax
)

ax.set_title(
    f"ACF of SARIMAX{best_order} + Fourier Residuals"
)

plt.tight_layout()
plt.show()

In [ ]:
#Compare residual diagnostics of the top models

candidate_orders = [
    (1, 0, 2),
    (3, 0, 3),
    (2, 0, 2),
    (3, 0, 1)
]

diagnostic_results = []

for order in candidate_orders:
    
    print(f"Checking SARIMAX{order}...")
    
    model = SARIMAX(
        train_cv,
        exog=X_train_cv,
        order=order,
        seasonal_order=(0, 0, 0, 0),
        enforce_stationarity=False,
        enforce_invertibility=False
    )
    
    fit = model.fit(disp=False)
    
    residuals_candidate = fit.resid.dropna()
    
    lb = acorr_ljungbox(
        residuals_candidate,
        lags=[7, 14, 30],
        return_df=True
    )
    
    diagnostic_results.append({
        "Order": str(order),
        "AIC": fit.aic,
        "BIC": fit.bic,
        "LB_pvalue_7": lb.loc[7, "lb_pvalue"],
        "LB_pvalue_14": lb.loc[14, "lb_pvalue"],
        "LB_pvalue_30": lb.loc[30, "lb_pvalue"]
    })

diagnostic_comparison = pd.DataFrame(diagnostic_results)

diagnostic_comparison

In [ ]:
# Visualize the residual ACF for the best alternative

best_order = (3, 0, 3)  # CHANGE this after we examine the table

best_model = SARIMAX(
    train_cv,
    exog=X_train_cv,
    order=best_order,
    seasonal_order=(0, 0, 0, 0),
    enforce_stationarity=False,
    enforce_invertibility=False
)

best_fit = best_model.fit(disp=False)

best_residuals = best_fit.resid.dropna()

print(best_fit.summary())

print("\nLjung-Box test:")
print(
    acorr_ljungbox(
        best_residuals,
        lags=[7, 14, 30],
        return_df=True
    )
)

In [ ]:
# Confirm the residual ACF

final_candidate_order = (3, 0, 1)

final_candidate_model = SARIMAX(
    train_cv,
    exog=X_train_cv,
    order=final_candidate_order,
    seasonal_order=(0, 0, 0, 0),
    enforce_stationarity=False,
    enforce_invertibility=False
)

final_candidate_fit = final_candidate_model.fit(disp=False)

final_candidate_residuals = final_candidate_fit.resid.dropna()

print(final_candidate_fit.summary())

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

plot_acf(
    final_candidate_residuals,
    lags=40,
    ax=ax
)

ax.set_title(
    "ACF of SARIMAX(3,0,1) + Fourier Residuals"
)

plt.tight_layout()
plt.show()

In [ ]:
# Final residual Ljung–Box test

final_ljung_box = acorr_ljungbox(
    final_candidate_residuals,
    lags=[7, 14, 30],
    return_df=True
)

print(final_ljung_box)

In [ ]:
train_final = temperature[
    temperature.index < "1990-01-01"
]

final_test = temperature[
    temperature.index >= "1990-01-01"
]

print(
    "Final training:",
    train_final.index.min(),
    "to",
    train_final.index.max(),
    len(train_final)
)

print(
    "Final test:",
    final_test.index.min(),
    "to",
    final_test.index.max(),
    len(final_test)
)

In [ ]:
# Create Fourier terms for the final training/test periods

X_train_final = fourier_all.loc[train_final.index]
X_test_final = fourier_all.loc[final_test.index]

print(X_train_final.shape)
print(X_test_final.shape)

In [ ]:
final_model = SARIMAX(
    train_final,
    exog=X_train_final,
    order=(3, 0, 1),
    seasonal_order=(0, 0, 0, 0),
    enforce_stationarity=False,
    enforce_invertibility=False
)

final_fit = final_model.fit(disp=False)

print(final_fit.summary())

In [ ]:
# Forecast the completely unseen 1990 data

final_forecast = final_fit.forecast(
    steps=len(final_test),
    exog=X_test_final
)

final_forecast.index = final_test.index

In [ ]:
final_test_metrics = evaluate_model(
    final_test,
    final_forecast
)

print("FINAL TEST PERFORMANCE")
print(final_test_metrics)

In [ ]:
# Plot the final test forecast

plt.figure(figsize=(15, 6))

plt.plot(
    final_test.index,
    final_test,
    label="Actual"
)

plt.plot(
    final_forecast.index,
    final_forecast,
    label="SARIMAX(3,0,1) + Fourier"
)

plt.title(
    "Final Test: Actual vs SARIMAX(3,0,1) + Fourier Forecast"
)

plt.xlabel("Date")
plt.ylabel("Minimum Temperature (°C)")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Calculate prediction intervals

forecast_result = final_fit.get_forecast(
    steps=len(final_test),
    exog=X_test_final
)

forecast_mean = forecast_result.predicted_mean
forecast_ci = forecast_result.conf_int()

In [ ]:
print(forecast_ci.head())

In [ ]:
plt.figure(figsize=(15, 6))

plt.plot(
    final_test.index,
    final_test,
    label="Actual"
)

plt.plot(
    forecast_mean.index,
    forecast_mean,
    label="Forecast"
)

plt.fill_between(
    forecast_ci.index,
    forecast_ci.iloc[:, 0],
    forecast_ci.iloc[:, 1],
    alpha=0.2,
    label="95% prediction interval"
)

plt.title(
    "SARIMAX(3,0,1) + Fourier: Final Forecast with 95% Prediction Interval"
)

plt.xlabel("Date")
plt.ylabel("Minimum Temperature (°C)")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Plot the final model performance

plt.figure(figsize=(15, 6))

plt.plot(
    final_test.index,
    final_test,
    label="Actual"
)

plt.plot(
    final_forecast.index,
    final_forecast,
    label="SARIMAX(3,0,1) + Fourier"
)

plt.title(
    "Final Test Performance: 1990 Daily Minimum Temperature"
)

plt.xlabel("Date")
plt.ylabel("Minimum Temperature (°C)")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
forecast_result = final_fit.get_forecast(
    steps=len(final_test),
    exog=X_test_final
)

forecast_mean = forecast_result.predicted_mean
forecast_ci = forecast_result.conf_int()

In [ ]:
print(forecast_ci.head())

In [ ]:
plt.figure(figsize=(15, 6))

plt.plot(
    final_test.index,
    final_test,
    label="Actual"
)

plt.plot(
    forecast_mean.index,
    forecast_mean,
    label="Forecast"
)

plt.fill_between(
    forecast_ci.index,
    forecast_ci.iloc[:, 0],
    forecast_ci.iloc[:, 1],
    alpha=0.2,
    label="95% Prediction Interval"
)

plt.title(
    "SARIMAX(3,0,1) + Fourier: Final Forecast with 95% Prediction Interval"
)

plt.xlabel("Date")
plt.ylabel("Minimum Temperature (°C)")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
final_test_residuals = final_test - final_forecast

print(final_test_residuals.describe())

In [ ]:
mean_error = final_test_residuals.mean()

print("Mean forecast error:", mean_error)

In [ ]:
# Calculate percentage improvement

seasonal_naive_rmse = 3.650761
seasonal_naive_mae = 2.867397

final_rmse = final_test_metrics["RMSE"]
final_mae = final_test_metrics["MAE"]

rmse_improvement = (
    (seasonal_naive_rmse - final_rmse)
    / seasonal_naive_rmse
) * 100

mae_improvement = (
    (seasonal_naive_mae - final_mae)
    / seasonal_naive_mae
) * 100

print(f"RMSE improvement: {rmse_improvement:.2f}%")
print(f"MAE improvement: {mae_improvement:.2f}%")

In [ ]:
# Actual future forecast for 1981–1990 using the final model

full_data = temperature.copy()

X_full = fourier_all.loc[full_data.index]

final_full_model = SARIMAX(
    full_data,
    exog=X_full,
    order=(3, 0, 1),
    seasonal_order=(0, 0, 0, 0),
    enforce_stationarity=False,
    enforce_invertibility=False
)

final_full_fit = final_full_model.fit(disp=False)

print(final_full_fit.summary())

In [ ]:
forecast_days = 365

future_index = pd.date_range(
    start=temperature.index.max() + pd.Timedelta(days=1),
    periods=forecast_days,
    freq="D"
)

print("Forecast starts:", future_index.min())
print("Forecast ends:", future_index.max())

In [ ]:
# Create Fourier terms for the future

all_index_for_forecast = temperature.index.append(
    future_index
)

fourier_future_all = create_fourier_terms(
    all_index_for_forecast,
    period=365.25,
    K=5
)

X_future = fourier_future_all.loc[future_index]

print(X_future.shape)

In [ ]:
future_result = final_full_fit.get_forecast(
    steps=forecast_days,
    exog=X_future
)

future_forecast = future_result.predicted_mean
future_ci = future_result.conf_int()

In [ ]:
future_forecast.head(10)

In [ ]:
future_forecast.describe()

In [ ]:
# Plot the historical data + one-year forecast

plt.figure(figsize=(16, 7))

# Last 2 years of historical data
recent_history = temperature.tail(730)

plt.plot(
    recent_history.index,
    recent_history,
    label="Historical"
)

plt.plot(
    future_forecast.index,
    future_forecast,
    label="365-Day Forecast"
)

plt.fill_between(
    future_ci.index,
    future_ci.iloc[:, 0],
    future_ci.iloc[:, 1],
    alpha=0.2,
    label="95% Prediction Interval"
)

plt.axvline(
    temperature.index.max(),
    linestyle="--"
)

plt.title(
    "365-Day Forecast of Daily Minimum Temperature"
)

plt.xlabel("Date")
plt.ylabel("Minimum Temperature (°C)")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# monthly forecast summary

future_forecast_df = pd.DataFrame({
    "Forecast": future_forecast,
    "Lower_95": future_ci.iloc[:, 0],
    "Upper_95": future_ci.iloc[:, 1]
})

future_forecast_df["Month"] = future_forecast_df.index.month

monthly_forecast = (
    future_forecast_df
    .groupby("Month")
    .agg(
        Mean_Forecast=("Forecast", "mean"),
        Minimum_Forecast=("Forecast", "min"),
        Maximum_Forecast=("Forecast", "max")
    )
)

monthly_forecast.index = [
    "January", "February", "March", "April",
    "May", "June", "July", "August",
    "September", "October", "November", "December"
]

monthly_forecast

In [ ]:
# final forecast table

final_forecast_table = future_forecast_df.copy()

final_forecast_table.to_csv(
    "SARIMAX_Fourier_365_day_forecast.csv"
)

print("Forecast saved successfully.")

In [ ]:
# final model-results table

final_results = pd.DataFrame({
    "Model": [
        "Naïve",
        "Seasonal Naïve",
        "ARIMA(1,0,2)",
        "Holt-Winters",
        "SARIMAX(3,0,1) + Fourier"
    ],
    "MAE": [
        3.323562,
        2.867397,
        3.198161,
        3.542510,
        final_test_metrics["MAE"]
    ],
    "RMSE": [
        3.991597,
        3.650761,
        3.862486,
        4.204320,
        final_test_metrics["RMSE"]
    ]
})

final_results

In [ ]:
# Create the final forecast dataframe

final_forecast_df = pd.DataFrame({
    "Date": future_forecast.index,
    "Forecast_Temperature": future_forecast.values,
    "Lower_95_CI": future_ci.iloc[:, 0].values,
    "Upper_95_CI": future_ci.iloc[:, 1].values
})

final_forecast_df["Month"] = final_forecast_df["Date"].dt.month_name()

final_forecast_df.head()

In [ ]:
monthly_forecast = (
    final_forecast_df
    .groupby("Month")
    .agg(
        Mean_Forecast=("Forecast_Temperature", "mean"),
        Minimum_Forecast=("Forecast_Temperature", "min"),
        Maximum_Forecast=("Forecast_Temperature", "max")
    )
)

month_order = [
    "January", "February", "March", "April",
    "May", "June", "July", "August",
    "September", "October", "November", "December"
]

monthly_forecast = monthly_forecast.reindex(month_order)

monthly_forecast.round(2)

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(
    monthly_forecast.index,
    monthly_forecast["Mean_Forecast"],
    marker="o"
)

plt.title("Monthly Mean Forecast for 1991")
plt.xlabel("Month")
plt.ylabel("Forecast Minimum Temperature (°C)")
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
final_forecast_df["Interval_Width"] = (
    final_forecast_df["Upper_95_CI"]
    - final_forecast_df["Lower_95_CI"]
)

print(
    "Average 95% prediction interval width:",
    final_forecast_df["Interval_Width"].mean()
)

print(
    "Minimum interval width:",
    final_forecast_df["Interval_Width"].min()
)

print(
    "Maximum interval width:",
    final_forecast_df["Interval_Width"].max()
)

In [ ]:
final_residuals = final_full_fit.resid.dropna()

print(final_residuals.describe())

In [ ]:
final_ljung_box = acorr_ljungbox(
    final_residuals,
    lags=[7, 14, 30],
    return_df=True
)

print(final_ljung_box)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

plot_acf(
    final_residuals,
    lags=40,
    ax=ax
)

ax.set_title(
    "ACF of Final SARIMAX(3,0,1) + Fourier Residuals"
)

plt.tight_layout()
plt.show()

In [ ]:
final_results = pd.DataFrame({
    "Model": [
        "Naïve",
        "Seasonal Naïve",
        "ARIMA(1,0,2)",
        "Holt-Winters",
        "SARIMAX(3,0,1) + Fourier"
    ],
    "MAE": [
        3.323562,
        2.867397,
        3.198161,
        3.542510,
        final_test_metrics["MAE"]
    ],
    "RMSE": [
        3.991597,
        3.650761,
        3.862486,
        4.204320,
        final_test_metrics["RMSE"]
    ]
})

final_results["RMSE_Rank"] = (
    final_results["RMSE"]
    .rank(method="min")
    .astype(int)
)

final_results = final_results.sort_values(
    "RMSE"
).reset_index(drop=True)

final_results.round(3)

In [ ]:
plt.figure(figsize=(10, 6))

plt.bar(
    final_results["Model"],
    final_results["RMSE"]
)

plt.title("Forecasting Model Comparison")
plt.xlabel("Model")
plt.ylabel("RMSE (°C)")
plt.xticks(rotation=45, ha="right")

plt.tight_layout()
plt.show()

In [ ]:
seasonal_naive_rmse = 3.650761
seasonal_naive_mae = 2.867397

final_rmse = final_test_metrics["RMSE"]
final_mae = final_test_metrics["MAE"]

rmse_improvement = (
    (seasonal_naive_rmse - final_rmse)
    / seasonal_naive_rmse
) * 100

mae_improvement = (
    (seasonal_naive_mae - final_mae)
    / seasonal_naive_mae
) * 100

print(f"RMSE improvement: {rmse_improvement:.2f}%")
print(f"MAE improvement: {mae_improvement:.2f}%")

In [ ]:
final_forecast_df.to_csv(
    "SARIMAX_Fourier_365_day_forecast.csv",
    index=False
)

In [ ]:
final_results.to_csv(
    "forecast_model_comparison.csv",
    index=False
)

In [ ]:
monthly_forecast.to_csv(
    "monthly_temperature_forecast.csv"
)

In [ ]:
import os

for file in [
    "SARIMAX_Fourier_365_day_forecast.csv",
    "forecast_model_comparison.csv",
    "monthly_temperature_forecast.csv"
]:
    print(file, "→", os.path.exists(file))

In [ ]:
print("=" * 60)
print("FINAL TIME SERIES FORECASTING MODEL")
print("=" * 60)

print("Model: SARIMAX(3,0,1) + Fourier")
print("Fourier harmonics: 5")
print("Seasonal period: 365.25 days")
print()

print("FINAL TEST PERFORMANCE (1990)")
print(f"MAE:  {final_test_metrics['MAE']:.3f} °C")
print(f"RMSE: {final_test_metrics['RMSE']:.3f} °C")
print()

print("IMPROVEMENT OVER SEASONAL NAÏVE")
print(f"MAE improvement:  {mae_improvement:.2f}%")
print(f"RMSE improvement: {rmse_improvement:.2f}%")
print()

print("365-DAY FORECAST")
print(f"Mean forecast: {future_forecast.mean():.2f} °C")
print(f"Minimum forecast: {future_forecast.min():.2f} °C")
print(f"Maximum forecast: {future_forecast.max():.2f} °C")
print("=" * 60)

In [ ]:
test_error_df = pd.DataFrame({
    "Actual": final_test,
    "Forecast": final_forecast
})

test_error_df["Error"] = (
    test_error_df["Actual"]
    - test_error_df["Forecast"]
)

test_error_df["Absolute_Error"] = (
    test_error_df["Error"].abs()
)

largest_errors = (
    test_error_df
    .sort_values("Absolute_Error", ascending=False)
    .head(10)
)

largest_errors

In [ ]:
print("Mean Error:", test_error_df["Error"].mean())
print("Median Error:", test_error_df["Error"].median())

print("\nMean Absolute Error:",
      test_error_df["Absolute_Error"].mean())

print("\nMaximum Absolute Error:",
      test_error_df["Absolute_Error"].max())

In [ ]:
plt.figure(figsize=(9, 5))

plt.hist(
    test_error_df["Error"],
    bins=30,
    edgecolor="black"
)

plt.axvline(
    0,
    linestyle="--"
)

plt.title("Distribution of Final Test Forecast Errors")
plt.xlabel("Forecast Error (Actual − Forecast), °C")
plt.ylabel("Frequency")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 7))

plt.scatter(
    final_test,
    final_forecast,
    alpha=0.6
)

min_value = min(
    final_test.min(),
    final_forecast.min()
)

max_value = max(
    final_test.max(),
    final_forecast.max()
)

plt.plot(
    [min_value, max_value],
    [min_value, max_value],
    linestyle="--"
)

plt.xlabel("Actual Temperature (°C)")
plt.ylabel("Predicted Temperature (°C)")
plt.title("Actual vs Predicted Daily Minimum Temperature")

plt.tight_layout()
plt.show()

#### Streamlit 

In [ ]:
final_full_fit = final_full_model.fit(disp=False)

In [ ]:
import os

os.makedirs("models", exist_ok=True)

final_full_fit.save(
    "models/sarimax_fourier_model.pkl"
)

print("Model saved successfully.")

In [ ]:
import os

print(
    os.path.exists(
        "models/sarimax_fourier_model.pkl"
    )
)

In [ ]:
import os

os.makedirs("outputs", exist_ok=True)

final_forecast_df.to_csv(
    "outputs/SARIMAX_Fourier_365_day_forecast.csv",
    index=False
)

final_results.to_csv(
    "outputs/forecast_model_comparison.csv",
    index=False
)

monthly_forecast.to_csv(
    "outputs/monthly_temperature_forecast.csv"
)

print("All output files saved.")